Exploration et qualité des données sources

Ce notebook explore les données sources qui serviront de base au projet NYC Mobility & Weather. L’objectif est de comprendre leur structure, leur volumétrie et leur qualité avant de concevoir le pipeline et le modèle de données dans PostgreSQL.

L’exploration couvre trois étapes :

1. **Profiler les données** en examinant le nombre de lignes, les types de colonnes, la période couverte, les valeurs manquantes et les distributions.
2. **Évaluer la qualité** en repérant les doublons potentiels, les dates incohérentes et les valeurs atypiques, puis en mesurant la fréquence de ces anomalies.
3. **Documenter les décisions** en s’appuyant sur les observations pour choisir les types SQL, définir la granularité des tables et proposer des règles de validation, de correction ou de filtrage.

In [15]:
import pandas as pd
import requests
import plotly.express as px


In [2]:
#Load and profile the Yellow Taxi data.
df_taxi = pd.read_parquet(
    r'../src/data/raw/yellow_taxi/yellow_tripdata_2026-01.parquet',
    engine='pyarrow',
)
display(df_taxi.head(20))

df_taxi_global_analysis=df_taxi.iloc[0:10000]
df_taxi_global_analysis.to_csv('df_taxi_global_analysis.csv')

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75
5,2,2026-01-01 00:47:11,2026-01-01 01:00:47,2.0,2.33,1.0,N,144,137,1,14.2,1.00,0.5,4.99,0.0,1.0,24.94,2.5,0.0,0.75
6,1,2026-01-01 00:17:54,2026-01-01 00:28:32,1.0,1.30,1.0,N,142,50,2,11.4,4.25,0.5,0.00,0.0,1.0,17.15,2.5,0.0,0.75
7,1,2026-01-01 00:34:28,2026-01-01 00:59:05,0.0,2.90,1.0,N,50,234,1,22.6,4.25,0.5,5.65,0.0,1.0,34.00,2.5,0.0,0.75
8,2,2026-01-01 00:34:14,2026-01-01 01:11:58,1.0,5.34,1.0,N,161,45,1,37.3,1.00,0.5,8.61,0.0,1.0,51.66,2.5,0.0,0.75
9,2,2026-01-01 00:41:07,2026-01-01 00:50:42,3.0,1.83,1.0,N,237,263,1,10.7,1.00,0.5,2.36,0.0,1.0,18.06,2.5,0.0,0.00


In [3]:
df_info=df_taxi.describe()
df_info=df_info.reset_index()
df_info.to_csv('df_info.csv', index=False)

In [4]:
#analyse valeurs manquantes
nb_manquantes = df_taxi.isna().sum()

profil_manquantes = pd.DataFrame({
    "nb_manquantes": nb_manquantes,
    "pct_manquantes": (nb_manquantes / len(df_taxi) * 100).round(2),
}).sort_values("nb_manquantes", ascending=False)

display(profil_manquantes)

,nb_manquantes,pct_manquantes
Airport_fee,1088058,29.21
passenger_count,1088058,29.21
congestion_surcharge,1088058,29.21
RatecodeID,1088058,29.21
store_and_fwd_flag,1088058,29.21
VendorID,0,0.00
mta_tax,0,0.00
total_amount,0,0.00
improvement_surcharge,0,0.00
tolls_amount,0,0.00


In [5]:
#analyse valeurs négatifs
df_taxi[df_taxi['tolls_amount']<0].head()
#print(df_taxi[df_taxi['tolls_amount']<0].count())

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
1204,2,2026-01-01 00:41:06,2026-01-01 01:16:09,1.0,13.31,1.0,N,107,182,4,-54.10,-1.0,-0.5,0.0,-6.94,-1.0,-66.79,-2.5,0.00,-0.75
3959,2,2026-01-01 00:33:43,2026-01-01 01:26:53,4.0,17.03,5.0,N,161,265,3,-95.00,0.0,0.0,0.0,-14.06,-1.0,-110.81,0.0,0.00,-0.75
4826,2,2026-01-01 00:33:07,2026-01-01 00:47:35,1.0,3.45,4.0,N,68,265,4,-25.40,-1.0,0.0,0.0,-14.06,-1.0,-42.21,0.0,0.00,-0.75
4872,2,2026-01-01 00:14:31,2026-01-01 00:27:26,3.0,7.72,5.0,N,138,42,2,-68.99,-5.0,0.0,0.0,-21.00,-1.0,-97.74,0.0,-1.75,0.00
9119,2,2026-01-01 01:57:45,2026-01-01 02:13:04,1.0,4.18,4.0,N,48,265,4,-28.90,-1.0,0.0,0.0,-14.06,-1.0,-45.71,0.0,0.00,-0.75


In [6]:
df_taxi[df_taxi['RatecodeID']==99].count()
print(df_taxi[df_taxi['RatecodeID'] == 99].head())

      VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
642          1  2026-01-01 00:11:59   2026-01-01 01:07:53              1.0   
822          1  2026-01-01 00:11:09   2026-01-01 00:47:05              1.0   
837          1  2026-01-01 00:58:46   2026-01-01 01:26:34              1.0   
1361         1  2026-01-01 00:54:38   2026-01-01 01:54:59              1.0   
1529         1  2026-01-01 00:41:52   2026-01-01 01:43:46              1.0   

      trip_distance  RatecodeID store_and_fwd_flag  PULocationID  \
642             5.2        99.0                  N            91   
822             9.3        99.0                  N            25   
837             6.3        99.0                  N            61   
1361           16.0        99.0                  N           205   
1529            7.1        99.0                  N            25   

      DOLocationID  payment_type  fare_amount  extra  mta_tax  tip_amount  \
642            155             1         20.5

Les statistiques descriptives montrent que passenger_count, RatecodeID, congestion_surcharge et Airport_fee présentent chacune 29,21 % de valeurs manquantes. Comme ces variables sont pertinentes pour l’étude, les analyses qui les utilisent excluront les lignes où les champs nécessaires sont manquants. Par ailleurs soit 0,091 % du fichier, ont un total_amount négatif. Ils seront exclus du calcul du montant moyen facturé. Les données d’origine seront conservées dans la couche raw.




In [7]:
#analyse données wheather
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 40.7128,
    "longitude": -74.0060,
    "start_date": "2025-12-31",
    "end_date": "2026-02-01",
    "hourly": "temperature_2m,rain,snowfall,wind_speed_10m",
    "timezone": "America/New_York",
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()
data = response.json()

df_weather = pd.DataFrame(data["hourly"])
df_weather["time"] = pd.to_datetime(df_weather["time"])

print("Nombre de lignes :", len(df_weather))
print("Unités :", data["hourly_units"])

Nombre de lignes : 792
Unités : {'time': 'iso8601', 'temperature_2m': '°C', 'rain': 'mm', 'snowfall': 'cm', 'wind_speed_10m': 'km/h'}


In [8]:
print(df_weather.head())

                 time  temperature_2m  rain  snowfall  wind_speed_10m
0 2025-12-31 00:00:00            -3.7   0.0       0.0            13.2
1 2025-12-31 01:00:00            -3.4   0.0       0.0            13.8
2 2025-12-31 02:00:00            -3.4   0.0       0.0            14.0
3 2025-12-31 03:00:00            -3.4   0.0       0.0            13.2
4 2025-12-31 04:00:00            -3.3   0.0       0.0            13.2


In [9]:
pickup = pd.to_datetime(df_taxi["tpep_pickup_datetime"])
dropoff = pd.to_datetime(df_taxi["tpep_dropoff_datetime"])

df_taxi["year"] = pickup.dt.year
df_taxi["month"] = pickup.dt.month
df_taxi["day"] = pickup.dt.day
df_taxi["hour_of_day"] = pickup.dt.hour

df_taxi.iloc[0:1000].to_csv('df_taxi_with_hourly.csv', index=False)

In [10]:
weather_time = pd.to_datetime(df_weather["time"])

df_weather["year"] = weather_time.dt.year
df_weather["month"] = weather_time.dt.month
df_weather["day"] = weather_time.dt.day
df_weather["hour_of_day"] = weather_time.dt.hour
df_weather.to_csv('df_weather_with_hourly.csv', index=False)
print(df_weather[["time", "year", "month", "day", "hour_of_day"]].head(100))

                  time  year  month  day  hour_of_day
0  2025-12-31 00:00:00  2025     12   31            0
1  2025-12-31 01:00:00  2025     12   31            1
2  2025-12-31 02:00:00  2025     12   31            2
3  2025-12-31 03:00:00  2025     12   31            3
4  2025-12-31 04:00:00  2025     12   31            4
..                 ...   ...    ...  ...          ...
95 2026-01-03 23:00:00  2026      1    3           23
96 2026-01-04 00:00:00  2026      1    4            0
97 2026-01-04 01:00:00  2026      1    4            1
98 2026-01-04 02:00:00  2026      1    4            2
99 2026-01-04 03:00:00  2026      1    4            3

[100 rows x 5 columns]


In [11]:
#La modélisation repose sur une clé horaire (hour_key) qui identifie de manière unique chaque créneau dans la table météo et permet de lui rattacher les courses de taxi pour l’analyse.
def creer_cle_horaire(df):
    return (
        df["year"].astype("Int64").astype("string").str.zfill(4)
        + "-" + df["month"].astype("Int64").astype("string").str.zfill(2)
        + "-" + df["day"].astype("Int64").astype("string").str.zfill(2)
        + "-" + df["hour_of_day"].astype("Int64").astype("string").str.zfill(2)
    )

df_taxi["hour_key"] = creer_cle_horaire(df_taxi)
df_weather["hour_key"] = creer_cle_horaire(df_weather)

In [12]:
cle = df_weather["hour_key"]

print("Clés uniques :", cle.is_unique)
print("Clés manquantes :", cle.isna().sum())

# Affiche toutes les lignes qui partagent une clé.
display(df_weather.loc[cle.duplicated(keep=False)].sort_values("hour_key"))

Clés uniques : True
Clés manquantes : 0


,time,temperature_2m,rain,snowfall,wind_speed_10m,year,month,day,hour_of_day,hour_key


In [13]:
df_taxi_weather = df_taxi.merge(
    df_weather[["hour_key", "temperature_2m", "rain", "snowfall", "wind_speed_10m"]],
    on="hour_key",
    how="left",
    validate="many_to_one",
    indicator="weather_match",
)

display(df_taxi_weather["weather_match"].value_counts())

weather_match
both          3724889
left_only           0
right_only          0
Name: count, dtype: int64

In [16]:
debut = pd.Timestamp("2026-01-01")
fin = pd.Timestamp("2026-02-01")

# Total quotidien des passagers déclarés.
depart = pd.to_datetime(df_taxi_weather["tpep_pickup_datetime"])
taxi = df_taxi_weather.loc[depart.ge(debut) & depart.lt(fin), ["passenger_count"]].copy()
taxi["jour"] = depart.loc[taxi.index].dt.normalize()

passagers = (
    taxi.groupby("jour")["passenger_count"]
    .sum(min_count=1)
    .rename("passagers_declares")
)

# Une ligne météo par heure, avec pluie et neige déjà alignées.
meteo = df_weather.copy()
meteo["time"] = pd.to_datetime(meteo["time"])
meteo = meteo.loc[meteo["time"].ge(debut) & meteo["time"].lt(fin)].copy()
meteo["jour"] = meteo["time"].dt.normalize()

meteo_jour = meteo.groupby("jour").agg(
    temperature_moyenne=("temperature_2m", "mean"),
    vent_moyen=("wind_speed_10m", "mean"),
    pluie_totale=("rain", lambda s: s.sum(min_count=1)),
    neige_totale=("snowfall", lambda s: s.sum(min_count=1)),
)

bilan = meteo_jour.join(passagers).reset_index()

variables = {
    "temperature_moyenne": "Température moyenne (°C)",
    "vent_moyen": "Vitesse moyenne du vent (km/h)",
    "pluie_totale": "Pluie cumulée (mm)",
    "neige_totale": "Neige cumulée (cm)",
}

for colonne, libelle in variables.items():
    fig = px.scatter(
        bilan,
        x=colonne,
        y="passagers_declares",
        hover_data=["jour"],
        labels={colonne: libelle, "passagers_declares": "Passagers déclarés"},
        title=f"Passagers déclarés par jour selon {libelle.lower()} — janvier 2026",
        template="plotly_white",
    )
    fig.show()